# 🌍 Impact des Crises & Réfugiés — Analyse de Données

Bienvenue dans ce notebook d'analyse de données.  
Ce document vise à démontrer, **preuves statistiques à l'appui**, l'impact direct des conflits armés (guerres civiles, invasions) sur les mouvements de populations à travers le monde.  
Les données officielles sont issues de la **Banque Mondiale**.

## 📦 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Pour afficher Plotly dans Jupyter
import plotly.io as pio
pio.renderers.default = 'notebook'

## 📂 2. Chargement des données

In [ ]:
chemin_dossier = r"C:\Users\Asus\Downloads\Compressed\API_SM.POP.REFG.OR_DS57_en_csv_v2_7413"

df = pd.read_csv(chemin_dossier + r"\APP.csv", skiprows=4)
dc = pd.read_csv(chemin_dossier + r"\Metadata_Country.csv")

print(f"✅ Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

## 🔧 3. Transformation des données (Melt + Merge)

In [ ]:
# Transformation de 'df' (Melt) : on rassemble toutes les colonnes d'années en une seule colonne 'Annee'
df_melted = df.melt(
    id_vars=['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code'],
    var_name='Annee',
    value_name='Nombre_Refugies'
)

# On s'assure que la colonne année ne contient que des nombres et on la convertit en entier
df_melted = df_melted[df_melted['Annee'].str.isnumeric() == True]
df_melted['Annee'] = df_melted['Annee'].astype(int)

# Fusion avec les métadonnées pays pour ajouter la Région et le groupe de revenu
df_final = pd.merge(
    df_melted,
    dc[['Country Code', 'Region', 'IncomeGroup']],
    on='Country Code',
    how='inner'
)

# Nettoyage : suppression des lignes sans données + renommage
df_final = df_final.dropna(subset=['Nombre_Refugies', 'Region'])
df_final = df_final.rename(columns={'Country Name': 'Pays', 'Country Code': 'Code_ISO'})

print(f"✅ Dataset final : {df_final.shape[0]} lignes")
df_final.head(10)

## 🗂️ 4. Préparation : focus sur la période récente (depuis 1990)

In [ ]:
df_recent = df_final[df_final['Annee'] >= 1990].copy()
annee_max = df_recent['Annee'].max()
df_map = df_recent[df_recent['Annee'] == annee_max].copy()

annee_actuelle = annee_max
annee_passe = annee_actuelle - 10
df_actuel = df_recent[df_recent['Annee'] == annee_actuelle]
total_refugies_actuel = df_actuel['Nombre_Refugies'].sum()

print(f"📅 Année la plus récente dans les données : {annee_max}")
print(f"👥 Total de réfugiés en {annee_actuelle} : {total_refugies_actuel:,.0f}")

## 📊 5. KPI — Nombre total de réfugiés dans le monde

In [ ]:
fig_kpi = go.Figure(go.Indicator(
    mode="number",
    value=total_refugies_actuel,
    title={
        "text": f"<span style='font-size:30px'>Nombre total de réfugiés dans le monde ({annee_actuelle})</span>"
                "<br><span style='font-size:14px;color:gray'>Personnes ayant dû fuir leur pays</span>"
    },
    number={'valueformat': ' ', 'font': {'size': 80, 'color': '#d32f2f'}}
))

fig_kpi.show()

## 📈 6. Évolution globale par Région (Area Chart)

In [ ]:
df_region = df_recent.groupby(['Annee', 'Region'])['Nombre_Refugies'].sum().reset_index()

fig1 = px.area(
    df_region, x="Annee", y="Nombre_Refugies", color="Region",
    title="Évolution du volume de réfugiés par région d'origine (1990-2023)"
)
fig1.update_layout(template="plotly_white")
fig1.show()

## 📉 7. Courbe globale unique (ligne rouge)

In [ ]:
df_total = df_recent.groupby('Annee')['Nombre_Refugies'].sum().reset_index()

fig_courbe = px.line(
    df_total, x='Annee', y='Nombre_Refugies',
    title="L'évolution globale de la crise des réfugiés (1990 - Aujourd'hui)"
)
fig_courbe.update_traces(
    line_color='#d32f2f', line_width=4,
    fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
)
fig_courbe.update_layout(
    template="plotly_white",
    yaxis_title="Nombre total de réfugiés",
    xaxis_title="Année"
)
fig_courbe.show()

## 🌐 8. Carte Choroplèthe Animée

In [ ]:
# On borne la couleur au 98e centile pour éviter qu'un pays extrême écrase l'échelle
valeur_max = df_recent['Nombre_Refugies'].quantile(0.98)

fig2 = px.choropleth(
    df_recent.sort_values('Annee'),
    locations="Code_ISO",
    color="Nombre_Refugies",
    hover_name="Pays",
    animation_frame="Annee",
    color_continuous_scale="YlOrRd",
    title="Carte animée : Évolution mondiale des pays d'origine (1990-2023)",
    range_color=[0, valeur_max]
)
fig2.show()

## 🗺️ 9. Treemap — Répartition hiérarchique (Monde → Région → Pays)

In [ ]:
fig3 = px.treemap(
    df_map[df_map['Nombre_Refugies'] > 0],
    path=[px.Constant("Monde"), 'Region', 'Pays'],
    values='Nombre_Refugies',
    color='Nombre_Refugies',
    color_continuous_scale='Reds',
    title=f"Répartition des réfugiés par Région et Pays d'origine (Année {annee_max})"
)
fig3.show()

## 🏆 10. Top 15 des pays d'origine

In [ ]:
top_15 = df_map.nlargest(15, 'Nombre_Refugies')

fig4 = px.bar(
    top_15, x='Nombre_Refugies', y='Pays', orientation='h', color='Region',
    title=f"Top 15 des pays d'origine avec le plus de réfugiés (Année {annee_max})",
    text_auto='.2s'
)
fig4.update_layout(yaxis={'categoryorder': 'total ascending'}, template="plotly_white")
fig4.show()

## 🍩 11. Donut Chart — Répartition par continent

In [ ]:
fig_donut = px.pie(
    df_actuel, values='Nombre_Refugies', names='Region', hole=0.5,
    title=f"D'où viennent les réfugiés aujourd'hui ? (Répartition par continent en {annee_actuelle})"
)
fig_donut.update_traces(textposition='inside', textinfo='percent+label', showlegend=False)
fig_donut.show()

## ⚖️ 12. Comparaison Avant / Après (il y a 10 ans vs aujourd'hui)

In [ ]:
# Top 5 des pays les plus représentés aujourd'hui
top5_pays = df_actuel.nlargest(5, 'Nombre_Refugies')['Pays'].tolist()

df_compare = df_recent[
    (df_recent['Annee'].isin([annee_passe, annee_actuelle])) &
    (df_recent['Pays'].isin(top5_pays))
].copy()
df_compare['Annee'] = df_compare['Annee'].astype(str)

fig_avant_apres = px.bar(
    df_compare, x='Pays', y='Nombre_Refugies', color='Annee', barmode='group',
    title=f"L'explosion des crises : Comparaison {annee_passe} vs {annee_actuelle} (Top 5 actuel)",
    color_discrete_map={str(annee_passe): '#90caf9', str(annee_actuelle): '#1565c0'},
    text_auto='.2s'
)
fig_avant_apres.update_layout(template="plotly_white", yaxis_title="Nombre de réfugiés")
fig_avant_apres.show()

## ⚔️ 13. Impact direct des conflits armés — Courbe multi-séries annotée

In [ ]:
pays_cibles = ["Syrian Arab Republic", "Ukraine", "Afghanistan", "Congo, Dem. Rep."]
df_lignes = df_recent[df_recent['Pays'].isin(pays_cibles)].copy()

fig_lignes = px.line(
    df_lignes, x='Annee', y='Nombre_Refugies', color='Pays',
    title="Évolution comparée des crises majeures",
    labels={"Annee": "Année", "Nombre_Refugies": "Nombre de Réfugiés"}
)

# Annotation : début de la guerre en Syrie (2011)
val_syrie_df = df_lignes[(df_lignes['Pays'] == "Syrian Arab Republic") & (df_lignes['Annee'] == 2011)]
val_syrie = val_syrie_df['Nombre_Refugies'].max() if not val_syrie_df.empty else 0
fig_lignes.add_annotation(
    x=2011, y=val_syrie,
    text="Début de la guerre<br>en Syrie (2011)",
    showarrow=True, arrowhead=2, ax=-40, ay=-40
)

# Annotation : invasion de l'Ukraine (2022)
val_ukraine_df = df_lignes[(df_lignes['Pays'] == "Ukraine") & (df_lignes['Annee'] == 2022)]
val_ukraine = val_ukraine_df['Nombre_Refugies'].max() if not val_ukraine_df.empty else 6000000
fig_lignes.add_annotation(
    x=2022, y=val_ukraine,
    text="Invasion de<br>l'Ukraine (2022)",
    showarrow=True, arrowhead=2, ax=-50, ay=30
)

fig_lignes.update_layout(template="plotly_white")
fig_lignes.show()

## 🔀 14. Diagramme de Sankey — Origine des flux (Monde → Région → Pays)

In [ ]:
df_sankey = df_map[df_map['Nombre_Refugies'] > 0].copy()

# Pour lisibilité : Top 15 des pays, le reste regroupé
top15_pays = df_sankey.nlargest(15, 'Nombre_Refugies')['Pays'].tolist()
df_sankey['Pays_Filtre'] = df_sankey['Pays'].apply(
    lambda x: x if x in top15_pays else "Autres pays de la région"
)

# Niveau 1 : Monde → Région
flux_monde_region = df_sankey.groupby('Region')['Nombre_Refugies'].sum().reset_index()
flux_monde_region['Source'] = 'Monde'
flux_monde_region.rename(columns={'Region': 'Cible', 'Nombre_Refugies': 'Valeur'}, inplace=True)

# Niveau 2 : Région → Pays
flux_region_pays = df_sankey.groupby(['Region', 'Pays_Filtre'])['Nombre_Refugies'].sum().reset_index()
flux_region_pays.rename(columns={'Region': 'Source', 'Pays_Filtre': 'Cible', 'Nombre_Refugies': 'Valeur'}, inplace=True)

# Assemblage
sankey_data = pd.concat([flux_monde_region, flux_region_pays], ignore_index=True)

# Index des nœuds
all_nodes = list(pd.concat([sankey_data['Source'], sankey_data['Cible']]).unique())
node_indices = {node: i for i, node in enumerate(all_nodes)}

# Couleurs dynamiques
node_colors = []
for node in all_nodes:
    if node == 'Monde':
        node_colors.append('#212121')
    elif node in flux_monde_region['Cible'].values:
        node_colors.append('#1976d2')
    else:
        node_colors.append('#64b5f6')

fig_sankey = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15, thickness=20,
        line=dict(color="black", width=0.5),
        label=all_nodes,
        color=node_colors
    ),
    link=dict(
        source=[node_indices[src] for src in sankey_data['Source']],
        target=[node_indices[tgt] for tgt in sankey_data['Cible']],
        value=sankey_data['Valeur'],
        color="rgba(100, 181, 246, 0.4)"
    )
)])

fig_sankey.update_layout(
    title_text=f"Diagramme de Sankey : Répartition Mondiale des Pays d'Origine (Année {annee_max})",
    font_size=12,
    height=700
)
fig_sankey.show()

---
## ✅ Conclusion

Ce notebook démontre de manière statistique et visuelle la corrélation directe entre les conflits armés majeurs et les flux migratoires de réfugiés à l'échelle mondiale.  
**Sources** : Banque Mondiale — Indicateur `SM.POP.REFG.OR`